# Validar o ambiente do TCC no Google Colab

Este notebook testa o Google Meridian 2.1.0 em CPU. As instalações ficam em um ambiente temporário do Colab. O teste usa apenas imports e pequenos cálculos com dados artificiais.

1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, selecione **CPU** (sem acelerador) e a versão **2026.07**, com Python 3.12. Se a sessão estiver em Python 3.13, troque o runtime antes de continuar. [Versões do Colab](https://research.google.com/colaboratory/runtime-version-faq.html)
2. Execute as três células de código de cima para baixo e espere cada uma terminar. Se houver erro, pare na célula que falhou.

O teste termina com **AMBIENTE COLAB VALIDADO**. Envie ao Codex a saída da última célula. Se houver erro, envie o título da célula e a saída completa do erro. Ao reconectar a um novo runtime, comece pela primeira célula.


## 1. Conferir Python e clonar o TCC

A célula confere o Python do runtime e abre uma cópia temporária no commit auditado.

In [ ]:
import platform
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

checkout_verified = False
requirements_verified = False

if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Abra este notebook em um runtime Linux do Google Colab.")

try:
    from google import colab
except ImportError as error:
    raise RuntimeError("Abra este notebook no Google Colab.") from error

print("Python do Colab", platform.python_version(), flush=True)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Selecione o runtime Colab 2026.07 com Python 3.12 e rode esta célula novamente.")

git = shutil.which("git")
if git is None:
    raise RuntimeError("O executável do Git não foi encontrado no runtime.")

REPO_URL = "https://github.com/thiagossilva17/TCC_MMM_Google_Meridian.git"
REPO_COMMIT = "007f487d9f541c34a43395412c9637254bd65c04"
MERIDIAN_URL = "https://github.com/google/meridian.git"
MERIDIAN_COMMIT = "02111531f8661373aa7b6ba31c316c67d72d1dd2"
MERIDIAN_VERSION = "2.1.0"

work_dir = Path(tempfile.mkdtemp(prefix="tcc-meridian-", dir="/content"))
checkout = work_dir / "repositorio"
venv_dir = work_dir / "venv"
venv_python = venv_dir / "bin" / "python"

subprocess.run([git, "clone", "--quiet", "--no-checkout", "--no-tags", REPO_URL, str(checkout)], check=True)
subprocess.run([git, "checkout", "--quiet", "--detach", REPO_COMMIT], cwd=checkout, check=True)
actual_commit = subprocess.check_output([git, "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
if actual_commit != REPO_COMMIT:
    raise RuntimeError(f"Commit do TCC inesperado ({actual_commit}).")

checkout_verified = True
print("Commit do TCC confirmado", actual_commit)


## 2. Instalar e conferir as dependências

A célula instala os requisitos do commit auditado em um ambiente isolado e confere as versões, a compatibilidade entre pacotes e a origem do Meridian.

In [ ]:
if not globals().get("checkout_verified", False):
    raise RuntimeError("Conclua a célula 1 antes de instalar as dependências.")

requirements_verified = False

# O pip do Colab instala no ambiente indicado por --python.
# Isso permite criar o ambiente mesmo quando a imagem não fornece ensurepip.
subprocess.run([sys.executable, "-I", "-m", "venv", "--without-pip", str(venv_dir)], check=True)
pip = [
    sys.executable, "-I", "-m", "pip", "--isolated", "--disable-pip-version-check",
    "--python", str(venv_dir),
]
subprocess.run([*pip, "install", "--no-input", "-r", "requirements.txt"], cwd=checkout, check=True)
subprocess.run([*pip, "check"], cwd=checkout, check=True)

verify_installation = '''
import importlib.metadata as metadata
import json
import sys
from pathlib import Path

from packaging.requirements import Requirement

expected_url, expected_commit, expected_version = sys.argv[1:]
if sys.version_info[:2] != (3, 12) or sys.prefix == sys.base_prefix:
    raise RuntimeError("O interpretador isolado esperado é Python 3.12.")

for filename in ("requirements-temporal.txt", "constraints-tested.txt"):
    verified = 0
    for line in Path(filename).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        requirement = Requirement(line)
        if requirement.marker and not requirement.marker.evaluate():
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            if filename == "constraints-tested.txt":
                continue
            raise
        if not requirement.specifier.contains(installed, prereleases=True):
            raise RuntimeError(f"{requirement} é exigido. Versão instalada de {requirement.name}: {installed}.")
        verified += 1
    print(f"{filename} — {verified} versões conferidas.")

meridian = metadata.distribution("google-meridian")
origin = json.loads(meridian.read_text("direct_url.json") or "{}")
if meridian.version != expected_version:
    raise RuntimeError(f"Versão inesperada do Meridian ({meridian.version}).")
if origin.get("url") != expected_url or origin.get("vcs_info", {}).get("commit_id") != expected_commit:
    raise RuntimeError("A origem do Meridian instalado diverge do commit fixado.")

print("Versão e commit do Meridian confirmados", meridian.version, expected_commit)
'''
subprocess.run(
    [
        str(venv_python), "-I", "-B", "-u", "-c", verify_installation,
        MERIDIAN_URL, MERIDIAN_COMMIT, MERIDIAN_VERSION,
    ],
    cwd=checkout,
    check=True,
)
requirements_verified = True


## 3. Testar as bibliotecas em CPU

A célula mostra as versões, testa pequenos cálculos em CPU e carrega os componentes do Meridian usados pela EDA. Copie esta saída para compartilhar o resultado com o Codex.

In [ ]:
import os

if not globals().get("requirements_verified", False):
    raise RuntimeError("Conclua a célula 2 antes do diagnóstico.")

# Cada biblioteca é testada em um processo novo do ambiente isolado.
test_environment = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": "-1",
    "JAX_PLATFORMS": "cpu",
    "MERIDIAN_BACKEND": "jax",
    "MPLCONFIGDIR": str(work_dir / "matplotlib"),
    "TF_CPP_MIN_LOG_LEVEL": "2",
}

checks = {
    "Python": '''
import platform
import sys

assert sys.version_info[:2] == (3, 12) and sys.prefix != sys.base_prefix
print("Python isolado", platform.python_version(), "|", platform.system(), platform.machine())
''',
    "JAX": '''
import jax
import jax.numpy as jnp

devices = jax.devices()
assert devices and all(device.platform == "cpu" for device in devices)
assert int(jnp.arange(4).sum()) == 6
print("JAX", jax.__version__, "| dispositivos", devices)
''',
    "dm-tree": '''
from importlib.metadata import version

import tree

assert tree.flatten({"a": [1, 2]}) == [1, 2]
print("dm-tree", version("dm-tree"))
''',
    "TensorFlow": '''
import tensorflow as tf

devices = tf.config.list_physical_devices("CPU")
assert devices and int(tf.reduce_sum([1, 2, 3]).numpy()) == 6
print("TensorFlow", tf.__version__, "| dispositivos", devices)
''',
    "Meridian": '''
from importlib.metadata import distribution
import json

import meridian
from meridian.backend.config import get_backend
from meridian.data import data_frame_input_data_builder
from meridian.model import model, spec
from meridian.model.eda import meridian_eda
from meridian.model.eda.eda_engine import stack_variables

backend = get_backend().value
assert backend == "jax"
origin = json.loads(distribution("google-meridian").read_text("direct_url.json"))
print("Meridian", meridian.__version__, "| backend", backend)
print("Commit do Meridian", origin["vcs_info"]["commit_id"])
print("Imports da API usada pela EDA OK")
''',
}

print("Commit do TCC", REPO_COMMIT, flush=True)
print("Dependências e commit do Meridian conferidos na célula 2.", flush=True)
failed = []
for name, check in checks.items():
    print(f"\n{name}", flush=True)
    result = subprocess.run(
        [str(venv_python), "-I", "-B", "-u", "-c", check],
        cwd=work_dir,
        env=test_environment,
    )
    if result.returncode:
        failed.append(name)

if failed:
    raise RuntimeError("Falha no diagnóstico de " + ", ".join(failed))

print("\nAMBIENTE COLAB VALIDADO — Meridian 2.1.0 em CPU.", flush=True)
